# Qwen Extraction Agent — Phase 1 Proving Test

This notebook demonstrates the smallest end-to-end slice of the system:

```
source config -> Qwen3-8B -> JSON action -> execute_action -> tool result
```

It imports the actual project modules — it does not reimplement anything here.
Run this on a Colab GPU runtime (Runtime > Change runtime type > T4 GPU or better).

## 1. Clone the repository and install dependencies

In [ ]:
# Replace with your actual repository URL after pushing to GitHub.
# !git clone https://github.com/<your-org>/<your-repo>.git
# %cd <your-repo>

!pip install -q -r requirements.txt

## 2. Check CUDA availability

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 3. Load Qwen3-8B in 4-bit

In [ ]:
from agent.model import QwenModelClient

model_client = QwenModelClient(model_name="Qwen/Qwen3-8B", load_in_4bit=True)
model_client.load()
print("Model loaded.")

## 4. Verify GPU memory usage

In [ ]:
if torch.cuda.is_available():
    print("Allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))
    print("Reserved (GB):", round(torch.cuda.memory_reserved() / 1e9, 2))

## 5. Load the source configuration

In [ ]:
from core.config_loader import load_source_config

source_config = load_source_config("config/sources/test_direct_download.yaml")
source_config

## 6. Build the tool registry and run the agent loop

In [ ]:
from tools.definitions import build_registry
from agent.loop import run_agent

registry = build_registry()
final_state = run_agent(source_config, registry, model_client, max_steps=10)
print("Final status:", final_state.status)

## 7-10. Inspect the generated action, parsed JSON, executed tool result

`run_agent` already performed steps 7-10 internally (generate -> parse -> execute_action ->
http_download). The full structured history is available on `final_state`.

In [ ]:
for record in final_state.tool_history:
    print("step", record.step, "action", record.action)
    print("  arguments:", record.arguments)
    print("  result:", record.result)

print("\nArtifacts written:", final_state.artifacts)
print("Finish reason:", final_state.finish_reason)